In [2]:
import gc
import os

import numpy as np
import pandas as pd
import torch
from chemprop import data, featurizers
from chemprop import nn as chemprop_nn
from chemprop.nn import BondMessagePassing, MeanAggregation, SumAggregation
from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from torch import nn

from benchmarks import MultiTargetScaler, PrintEpochMetricsCallback
from benchmarks.lightning_integration import ChemPropLightningModule

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k_folds = 5
target_properties = [
    "xtb_homo",
    "xtb_lumo",
    "xtb_hl_gap",
    "pm7_homo",
    "pm7_lumo",
    "pm7_hl_gap",
    "pm7_hof",
]
property_types = [
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "intensive",
    "extensive",
]
dataset = "PAC5-1"
model_path_template = f"../models/chemprop_1.1M-graph-[{dataset}]-fold_{{fold}}"
model_logs_dir = "../logs"
model_dataset_dir = "../dataset"
model_logs_template = f"chemprop_1.1M-graph-[{dataset}]-fold_{{fold}}"
num_tasks = len(target_properties)
loss_weights = torch.ones(num_tasks, device=device)

criterion = nn.MSELoss()

featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()


class DualReadoutMPNN(nn.Module):
    """
    Shared D-MPNN backbone, property-type-aware readout (mirrors the G-CNN):
      intensive targets -> mean aggregation (size-invariant)
      extensive target  -> sum  aggregation (size-extensive)
    """

    def __init__(self, property_types, d_h=512, depth=3, ffn_hidden=512, ffn_layers=2):
        super().__init__()
        self.property_types = list(property_types)
        self.n_int = self.property_types.count("intensive")
        self.n_ext = self.property_types.count("extensive")

        self.mp = BondMessagePassing(d_h=d_h, depth=depth)
        d = self.mp.output_dim  # Actual output width, not hardcoded

        self.mean_agg = MeanAggregation()
        self.sum_agg = SumAggregation()

        def make_head(n_out):
            layers = [
                nn.BatchNorm1d(d)
            ]  # MPNN(batch_norm=True): BN after aggregation
            in_dim = d
            for _ in range(ffn_layers - 1):
                layers += [
                    nn.Linear(in_dim, ffn_hidden),
                    nn.BatchNorm1d(ffn_hidden),
                    nn.SiLU(),
                ]
                in_dim = ffn_hidden
            layers += [nn.Linear(in_dim, n_out)]
            return nn.Sequential(*layers)

        if self.n_int:
            self.int_head = make_head(self.n_int)
        if self.n_ext:
            self.ext_head = make_head(self.n_ext)

    def forward(self, bmg, V_d=None, X_d=None):
        H = self.mp(bmg, V_d)  # (V, d)
        int_pred = None
        ext_pred = None
        if self.n_int:
            int_pred = self.int_head(self.mean_agg(H, bmg.batch))
        if self.n_ext:
            ext_pred = self.ext_head(self.sum_agg(H, bmg.batch))

        out, i, e = [], 0, 0
        for p in self.property_types:
            if p == "intensive":
                if int_pred is None:
                    raise RuntimeError(
                        "The model does not have a readout head for intensive properties."
                    )
                out.append(int_pred[:, i : i + 1])
                i += 1
            else:
                if ext_pred is None:
                    raise RuntimeError(
                        "The model does not have a readout head for extensive properties."
                    )
                out.append(ext_pred[:, e : e + 1])
                e += 1
        return torch.cat(out, dim=1)  # (B, n_tasks) in target order

Using device: cuda


In [7]:
fold_results = []

for fold in range(1, k_folds + 1):
    print(f"\n======[ Fold {fold}/{k_folds} ]======")

    train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    val_path = model_dataset_dir + f"/{dataset}/fold_{fold}_val.csv"

    train_df = pd.read_csv(train_path).dropna(subset=target_properties)
    val_df = pd.read_csv(val_path).dropna(subset=target_properties)

    train_smiles = train_df["smiles"].values
    train_targets = train_df[target_properties].values.astype(np.float32)

    val_smiles = val_df["smiles"].values
    val_targets = val_df[target_properties].values.astype(np.float32)

    scaler = MultiTargetScaler()
    scaler.fit(train_targets)

    scaled_train_targets = scaler.transform(train_targets)
    scaled_val_targets = scaler.transform(val_targets)

    train_data = [
        data.MoleculeDatapoint.from_smi(smi, torch.tensor(y, dtype=torch.float32))
        for smi, y in zip(train_smiles, scaled_train_targets)
    ]
    val_data = [
        data.MoleculeDatapoint.from_smi(smi, torch.tensor(y, dtype=torch.float32))
        for smi, y in zip(val_smiles, scaled_val_targets)
    ]

    # Chemprop annotates MoleculeDatapoint.from_smi with return type of its mixin class instead of MoleculeDatapoint
    train_dataset = data.MoleculeDataset(train_data, featurizer, n_workers=8)  # pyrefly: ignore[bad-argument-type]
    val_dataset = data.MoleculeDataset(val_data, featurizer, n_workers=8)  # pyrefly: ignore[bad-argument-type]

    train_loader = data.build_dataloader(
        train_dataset, batch_size=256, shuffle=True, pin_memory=True, num_workers=8
    )
    val_loader = data.build_dataloader(
        val_dataset, batch_size=256, shuffle=False, pin_memory=True, num_workers=8
    )

    mp = chemprop_nn.BondMessagePassing(d_h=512, depth=3)
    agg = chemprop_nn.MeanAggregation()
    ffn = chemprop_nn.RegressionFFN(
        input_dim=mp.output_dim,
        n_tasks=len(target_properties),
        hidden_dim=512,
        n_layers=2,
    )
    model = DualReadoutMPNN(
        property_types=property_types, d_h=512, depth=3, ffn_hidden=512, ffn_layers=2
    )
    lightning_model = ChemPropLightningModule(model, target_properties, scaler, lr=1e-3)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=100,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[checkpoint_callback, PrintEpochMetricsCallback(target_properties)],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )

    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_model_path = checkpoint_callback.best_model_path
    best_lightning_model = ChemPropLightningModule.load_from_checkpoint(
        best_model_path, model=model, property_names=target_properties, scaler=scaler
    )
    torch.save(best_lightning_model.model.state_dict(), save_path_dir + "/best.ckpt")

    del train_df, val_df
    del train_data, val_data
    del train_dataset, val_dataset
    del train_loader, val_loader
    del model
    del lightning_model
    del trainer
    del scaler

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")


======[ Fold 1/5 ]======


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.

Detected KeyboardInterrupt, attempting graceful shutdown ...


RuntimeError: Please call `iter(combined_loader)` first.

In [4]:
import numpy as np

metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {prop: {m: [] for m in metric_keys} for prop in target_properties},
    "test": {prop: {m: [] for m in metric_keys} for prop in target_properties},
}

for fold in range(1, k_folds + 1):
    print(f"\n======[ Evaluating fold {fold}/{k_folds} ]======")

    train_path = model_dataset_dir + f"/{dataset}/fold_{fold}_train.csv"
    test_path = model_dataset_dir + f"/{dataset}/fold_{fold}_test.csv"

    train_df = pd.read_csv(train_path).dropna(subset=target_properties)
    test_df = pd.read_csv(test_path).dropna(subset=target_properties)

    train_smiles = train_df["smiles"].values
    train_targets = train_df[target_properties].values.astype(np.float32)

    test_smiles = test_df["smiles"].values
    test_targets = test_df[target_properties].values.astype(np.float32)

    scaler = MultiTargetScaler()
    scaler.fit(train_targets)

    scaled_train_targets = scaler.transform(train_targets)
    scaled_test_targets = scaler.transform(test_targets)

    train_data = [
        data.MoleculeDatapoint.from_smi(smi, torch.tensor(y, dtype=torch.float32))
        for smi, y in zip(train_smiles, scaled_train_targets)
    ]
    test_data = [
        data.MoleculeDatapoint.from_smi(smi, torch.tensor(y, dtype=torch.float32))
        for smi, y in zip(test_smiles, scaled_test_targets)
    ]

    # Chemprop annotates MoleculeDatapoint.from_smi with return type of its mixin class instead of MoleculeDatapoint
    train_dset = data.MoleculeDataset(train_data, featurizer, n_workers=8)  # pyrefly: ignore[bad-argument-type]
    test_dset = data.MoleculeDataset(test_data, featurizer, n_workers=8)  # pyrefly: ignore[bad-argument-type]

    train_loader = data.build_dataloader(
        train_dset, batch_size=256, shuffle=False, pin_memory=True, num_workers=8
    )
    test_loader = data.build_dataloader(
        test_dset, batch_size=256, shuffle=False, pin_memory=True, num_workers=8
    )

    model = DualReadoutMPNN(
        property_types=property_types, d_h=512, depth=3, ffn_hidden=512, ffn_layers=2
    )
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = ChemPropLightningModule(model, target_properties, scaler)

    trainer = pl.Trainer(accelerator="auto", devices=1, logger=False)

    print("\n======[ Train set ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("\n======[ Test set ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for i, prop in enumerate(target_properties):
        print(f"======[ {prop} ]======")
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        all_results["train"][prop]["mae"].append(t_mae)
        all_results["train"][prop]["rmsd"].append(t_rmsd)
        all_results["train"][prop]["max"].append(t_max)
        all_results["train"][prop]["min"].append(t_min)
        all_results["train"][prop]["r2"].append(t_r2)

        all_results["test"][prop]["mae"].append(te_mae)
        all_results["test"][prop]["rmsd"].append(te_rmsd)
        all_results["test"][prop]["max"].append(te_max)
        all_results["test"][prop]["min"].append(te_min)
        all_results["test"][prop]["r2"].append(te_r2)

print("======[ Cross-validation results ]======")

for prop in target_properties:
    print(f"\n======[ {prop} ]======")

    t_stats = {
        m: (
            np.mean(all_results["train"][prop][m]),
            np.std(all_results["train"][prop][m], ddof=1),
        )
        for m in metric_keys
    }

    print(
        f" + Train -> "
        f"MAE: {t_stats['mae'][0]:.4f} ± {t_stats['mae'][1]:.4f}\t"
        f"RMSD: {t_stats['rmsd'][0]:.4f} ± {t_stats['rmsd'][1]:.4f}\t"
        f"Max: {t_stats['max'][0]:.4f} ± {t_stats['max'][1]:.4f}\t"
        f"Min: {t_stats['min'][0]:.4f} ± {t_stats['min'][1]:.4f}\t"
        f"R^2: {t_stats['r2'][0]:.4f} ± {t_stats['r2'][1]:.4f}"
    )

    te_stats = {
        m: (
            np.mean(all_results["test"][prop][m]),
            np.std(all_results["test"][prop][m], ddof=1),
        )
        for m in metric_keys
    }

    print(
        f" + Test -> "
        f"MAE: {te_stats['mae'][0]:.4f} ± {te_stats['mae'][1]:.4f}\t"
        f"RMSD: {te_stats['rmsd'][0]:.4f} ± {te_stats['rmsd'][1]:.4f}\t"
        f"Max: {te_stats['max'][0]:.4f} ± {te_stats['max'][1]:.4f}\t"
        f"Min: {te_stats['min'][0]:.4f} ± {te_stats['min'][1]:.4f}\t"
        f"R^2: {te_stats['r2'][0]:.4f} ± {te_stats['r2'][1]:.4f}"
    )


======[ Evaluating fold 1/5 ]======


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
D:\Coding\pac-gcnn\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'test_dataloader' to speed up the dataloader worker initialization.



======[ Train set ]======



Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

D:\Coding\pac-gcnn\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
